In [ ]:
!pip install langchain langchain-groq langchain-community

#### Learned From Appiled with Ai channel video


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("API key loaded successfully")
else:
    print("API key not found. Check your .env file.")

API key loaded successfully


In [6]:
from groq import Groq

client = Groq(api_key=api_key)

models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
allam-2-7b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
openai/gpt-oss-20b
openai/gpt-oss-120b
whisper-large-v3
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi


In [7]:
from langchain_groq import ChatGroq

LLAMA_MODEL = "openai/gpt-oss-20b"

llm = ChatGroq(
    api_key=api_key,
    model=LLAMA_MODEL,
    temperature=0
)

In [8]:
def simple_chat_without_memory(user_query):
    response = llm.invoke(user_query)
    return response.content

In [ ]:
response1 = simple_chat_without_memory(
    "I want to learn AI"
)

print(response1)

In [ ]:
response2 = simple_chat_without_memory(
    "Can you give me a learning plan?"
)

print(response2)

In [13]:
from langchain_core.messages import HumanMessage,AIMessage

conversation=[]

def simple_chat_with_memory(user_query):
    conversation.append(HumanMessage(content=user_query))
    response = llm.invoke(conversation)
    conversation.append(AIMessage(content=response.content))
    return response.content

In [ ]:
response1 = simple_chat_with_memory(
    "I want to learn AI"
)

print(response1)

In [ ]:
print(conversation)

In [ ]:
response2 = simple_chat_with_memory(
    "Can you give me a learning plan?"
)

print(response2)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate(messages=[
    ("system", "you are a helpful personal assistant"),
    ("human", "{user_query}")
])

chain = prompt | llm
response_chain = chain.invoke({"user_query": "I have deceided to learn GenAI"})
print(response_chain)

In [ ]:
response_chain = chain.invoke({"user_query":"Can you give me a learning plan?"})
print(response_chain)

In [24]:
prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful personal assistant"),
    ("placeholder", "{chat_history}"),
    ("human", "{user_query}")
])

chain_with_memory = prompt_with_memory | llm

In [25]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}
# Function to get user's conversation history based on the user's session id
def get_history(session_id: str):
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]
     

In [ ]:
from langchain_core.runnables import RunnableWithMessageHistory

# Runnable with message history enables us to run a specified chain and
# gets the user's history using a callback function.

chat_with_memory = RunnableWithMessageHistory(
    runnable=chain_with_memory,
    get_session_history=get_history,
    input_messages_key="user_query",
    history_messages_key="chat_history"
)

response = chat_with_memory.invoke({"user_query":"I want to learn AI"}, {"configurable": {"session_id": "user1"}})
print(response)





In [ ]:
response2 = chat_with_memory.invoke(
    {"user_query": "Can you give me a learning plan?"},
    config={"configurable": {"session_id": "user1"}}
)

print(response2.content)

In [ ]:
response3 = chat_with_memory.invoke(
    {"user_query": "Can you give me a learning plan?"},
    config={"configurable": {"session_id": "user2"}}
)

print(response3.content)

